<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Multi-Head-Attention/multi-head-attention-q5-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("multi-head-attention", ...)`


# Implement Attention from Scratch
### Problem Statement
Multi-Head Attention (MHA) is the bread-and-butter of the Transformer architecture. It enables the model to **jointly attend** to information from different representation subspaces at different positions.

Your goal is to implement MHA from scratch using PyTorch, simulating exactly what `torch.nn.MultiheadAttention` does — projecting Q, K, V for each head, computing attention weights, applying them to V, and concatenating the outputs across all heads.

---

### Requirements

1. **Linear Projections for Q, K, V**
   - Project input `q`, `k`, `v` into a total of `d_model` dimensions.
   - Split them into `num_heads` of `d_head = d_model // num_heads` each.

2. **Scaled Dot-Product Attention per Head**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply an optional `mask` before softmax.
   - Use the scores to weight `V`.

3. **Combine the Heads**
   - Concatenate the outputs of all heads.
   - Apply a final linear projection to restore the shape: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch’s Reference**
   - Test your output against `torch.nn.MultiheadAttention` using the same input tensors.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Make sure all tensors are reshaped properly when splitting and combining heads.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when heads and shape are aligned.

---

<details>
  <summary>💡 Hint</summary>

  - Use `.view()` and `.transpose()` to shape Q, K, V to `(batch_size, num_heads, seq_len, d_head)`.
  - Softmax should be applied over the **last dimension** (attention scores across sequence).
  - Use `.contiguous().view()` to flatten the multi-head outputs back into `(batch_size, seq_len, d_model)`.
  - Match PyTorch’s behavior using the same projections and batch-first format.

</details>

---

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

torch.Size([3, 4, 8])


In [84]:
import torch
import torch.nn as nn
import torch.nn.functional as F




def multi_head_attention(q, k, v, num_heads, d_model, mask=None):
    """
    Implements multi-head attention.

    Args:
        q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
        k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
        v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
        num_heads (int): Number of attention heads
        d_model (int): Total embedding dimension
        mask (Tensor, optional): Masking tensor for attention

    Returns:
        Tensor: Multi-head attention output of shape (batch_size, seq_len, d_model)
    """
    assert d_model%num_heads == 0, 'dimension not appropriate'

    d_head = d_model//num_heads

    batch_size, seq_len, _ = q.shape

    W_q = nn.Linear(d_model, d_model, bias=False).to(q.device)
    W_k = nn.Linear(d_model, d_model, bias=False).to(q.device)
    W_v = nn.Linear(d_model, d_model, bias=False).to(q.device)
    W_out = nn.Linear(d_model, d_model, bias=False).to(q.device)

    q = W_q(q)
    k = W_k(k)
    v = W_v(v)

    # query_heads = []
    # key_heads = []
    # value_heads = []
    # attn_scores = []

    # for i in range(0, d_model, h_dim):

    #   query_heads.append(q[:,:,i:i+h_dim])
    #   key_heads.append(k[:,:,i:i+h_dim])
    #   value_heads.append(v[:,:,i:i+h_dim])

    q = q.view(batch_size, seq_len, num_heads, d_head).transpose(1,2)
    k = k.view(batch_size, seq_len, num_heads, d_head).transpose(1,2)
    v = v.view(batch_size, seq_len, num_heads, d_head).transpose(1,2)

    scores = torch.matmul(q, k.transpose(-2,-1)) / torch.sqrt(torch.tensor(d_head, device=q.device))

    if mask is not None:
      scores = scores.masked_fill(mask==0, float('-inf'))

    scores = F.softmax(scores, dim = -1)

    attn_score = torch.matmul(scores, v)

    out = attn_score.transpose(1,2).contiguous().view(batch_size, seq_len, d_model)

    return W_out(out)

    # for i in range(num_heads):

    #   scores = torch.matmul(query_heads[i], key_heads[i].permute(0,2,1)) / torch.sqrt(torch.tensor(h_dim, dtype=torch.float32, device=q.device))

    #   if mask is not None:
    #     scores = scores.masked_fill(mask==False, float('-inf'))

    #   scores = F.softmax(scores, dim=-1)
    #   attn_score = torch.matmul(scores, value_heads[i])
    #   attn_scores.append(attn_score)

    # attn_scores_concat = torch.cat(attn_scores, dim=-1)

    # return W_out(attn_scores_concat)




In [87]:
# Testing on data & compare
output_custom = multi_head_attention(q, k, v, num_heads, d_model)
print(output_custom)

multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True)
output, _ = multihead_attn(q, k, v)
print(output)

# assert torch.allclose(output_custom, output, atol=1e-08, rtol=1e-05) # Check if they are close enough.


tensor([[[-0.2851,  0.0395,  0.2139,  0.2267, -0.1134,  0.1139,  0.1427,
           0.1708],
         [-0.2856,  0.0377,  0.2108,  0.2269, -0.1146,  0.1128,  0.1444,
           0.1676],
         [-0.2849,  0.0389,  0.2132,  0.2264, -0.1138,  0.1135,  0.1432,
           0.1699],
         [-0.2856,  0.0387,  0.2114,  0.2269, -0.1144,  0.1133,  0.1445,
           0.1688]],

        [[-0.2497,  0.0956,  0.1524,  0.2175, -0.0698,  0.0804,  0.1250,
           0.1520],
         [-0.2487,  0.0946,  0.1518,  0.2167, -0.0704,  0.0797,  0.1259,
           0.1509],
         [-0.2491,  0.0952,  0.1534,  0.2173, -0.0692,  0.0801,  0.1266,
           0.1521],
         [-0.2476,  0.0945,  0.1528,  0.2160, -0.0700,  0.0785,  0.1268,
           0.1505]],

        [[-0.2724,  0.0197,  0.1507,  0.1768, -0.1855,  0.1334,  0.1255,
           0.1596],
         [-0.2718,  0.0205,  0.1494,  0.1763, -0.1859,  0.1337,  0.1245,
           0.1598],
         [-0.2741,  0.0203,  0.1513,  0.1774, -0.1868,  0.1357,  0